# 🎬 OpenSource Clipping — Beginner-Friendly Config

Welcome! This notebook lets you generate viral short-form clips from any video with just a few clicks.

**How to use:**
1. Run **Cell 1 (Setup)** once to install everything.
2. Fill in the **⚙️ Configuration** form fields below.
3. Run **Cell 8 (Validate & Review)** to check your settings.
4. Run **Cell 9 (Run Pipeline)** to generate your clips.

> 💡 **Tip:** You only need to change the settings in the form fields. You do **not** need to edit the main code.


## ① Setup — Install dependencies & clone repo

Run this cell first. It mounts Google Drive (optional), clones the repository, installs Python packages, and restores cached AI models from Drive.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ═══════════════════════════════════════════════════════════════
import os, subprocess, sys, shutil
from pathlib import Path

# ── Detect Google Colab ──
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Mount Google Drive (optional, for persistent model cache) ──
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"⚠️ Drive mount skipped: {e}")

# ── System diagnostics ──
def print_sys_info():
    print("═" * 60)
    try:
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, check=False)
        if gpu.returncode == 0 and gpu.stdout.strip():
            name, used, total = gpu.stdout.strip().split(", ")
            print(f"🎮 GPU : {name} | VRAM {float(used):.0f}/{float(total):.0f} MB")
    except Exception as e:
        print(f"⚠️ GPU read error: {e}")
    try:
        mem_lines = open("/proc/meminfo").read().splitlines()
        mem = dict((line.split()[0].rstrip(":"), int(line.split()[1])) for line in mem_lines)
        total_gb = mem.get("MemTotal", 0) / 1024**2
        free_gb = mem.get("MemAvailable", mem.get("MemFree", 0)) / 1024**2
        print(f"🧠 RAM : {free_gb:.1f} GB free / {total_gb:.1f} GB total")
    except Exception as e:
        print(f"⚠️ RAM read error: {e}")
    try:
        stat = shutil.disk_usage("/content" if IN_COLAB else os.getcwd())
        print(f"💾 Disk: {stat.free/1024**3:.1f} GB free / {stat.total/1024**3:.1f} GB total")
    except Exception as e:
        print(f"⚠️ Disk read error: {e}")
    print("═" * 60)

print_sys_info()

# ── Clone / update repo ──
REPO_URL = "https://github.com/troublescope/Ashortai.git"
CLONE_DIR = "/content/opensource-clipping" if IN_COLAB else str(Path.home() / "opensource-clipping")
Path(CLONE_DIR).mkdir(parents=True, exist_ok=True)

if os.path.isdir(f"{CLONE_DIR}/.git"):
    print("📁 Repo exists — pulling latest…")
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only"], check=False)
else:
    print("⬇️  Cloning repo…")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)

os.chdir(CLONE_DIR)

# ── Install dependencies ──
print("⏳ Installing dependencies (this takes ~2-3 min)…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# ── Backward-compat patch: GOOGLE_API_KEY only required for Gemini provider ──
main_py = Path(CLONE_DIR) / "main.py"
main_text = main_py.read_text(encoding="utf-8")
old_check = '    if not cfg.api_key_gemini:'
new_check = '    if cfg.ai_provider == "gemini" and not cfg.api_key_gemini:'
if old_check in main_text and new_check not in main_text:
    main_py.write_text(main_text.replace(old_check, new_check), encoding="utf-8")
    print("🩹 Patched main.py: GOOGLE_API_KEY only required when provider is gemini.")
print("✅ Dependencies installed.")

# ── Verify torch CUDA ──
try:
    import torch
    print(f"🔥 PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   Device: {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"⚠️ PyTorch check skipped: {e}")

# ── Restore cached models from Drive ──
if IN_COLAB:
    try:
        DRIVE_CACHE = Path("/content/drive/MyDrive/osc_cache")
        DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
        for m in ["blaze_face_full_range.tflite", "face_yolov8m.pt", "face_yolov8n.pt", "face_yolov8s.pt"]:
            src, dst = DRIVE_CACHE / m, Path(CLONE_DIR) / m
            if src.exists() and not dst.exists():
                shutil.copy2(src, dst)
                print(f"📥 Restored {m} from Drive cache")
    except Exception as e:
        print(f"⚠️ Model cache restore skipped: {e}")

print_sys_info()
print("\n✅ Setup complete. Proceed to the Configuration section below.")


## ⚙️ Configuration

Use the form fields in the cells below to configure the pipeline. Each section explains what the settings do.

> 📝 **Important:** After changing any form field, run that cell again so the new value is saved.


### 📥 Input

Tell the pipeline where to get the video.

- **Video URL**: A direct link to a YouTube, TikTok, Instagram, or Google Drive video.
- **Local File Path**: Path to a video file already on this machine (e.g. `/content/my_video.mp4`).
- **Playlist URL**: Link to a YouTube playlist if you want to process multiple videos later.
- **Source Platform**: Select the platform that matches your URL.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — INPUT CONFIGURATION
# ═══════════════════════════════════════════════════════════════

# Paste a video URL here (YouTube, TikTok, Instagram, Google Drive)
VIDEO_URL = "" #@param {type:"string"}

# Or provide a local file path already on this machine
LOCAL_FILE_PATH = "" #@param {type:"string"}

# Playlist URL (for batch processing workflows)
PLAYLIST_URL = "" #@param {type:"string"}

# Which platform is the source video from?
SOURCE_PLATFORM = "youtube" #@param ["youtube", "tiktok", "instagram", "gdrive"]

# Maximum source download height. 'max' = best available quality.
SOURCE_HEIGHT = "max" #@param ["max", "1080", "1440", "2160"]


### 🤖 AI Settings

Choose the AI that will analyze the transcript and pick the best clip moments.

- **Provider**: `ollama` (recommended for cloud models), `gemini`, or `nvidia`.
- **Model**: The model name to call.
- **API Keys**: Required for the chosen provider.
  - `ollama` with `https://ollama.com` → set **Ollama API Key**.
  - `gemini` → set **Google API Key**.
  - `nvidia` → set **NVIDIA API Key**.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — AI CONFIGURATION
# ═══════════════════════════════════════════════════════════════

# Which AI provider should analyze the transcript?
AI_PROVIDER = "ollama" #@param ["gemini", "nvidia", "ollama"]

# Model name. For Ollama Cloud use 'gemini-3-flash-preview:cloud'.
AI_MODEL = "gemini-3-flash-preview:cloud" #@param {type:"string"}

# Gemini / Google API key (required if provider is 'gemini' or used as fallback)
GOOGLE_API_KEY = "" #@param {type:"string"}

# Ollama / OpenAI-compatible API key (required for Ollama Cloud)
OLLAMA_API_KEY = "" #@param {type:"string"}

# NVIDIA API key (only needed if provider is 'nvidia')
NVIDIA_API_KEY = "" #@param {type:"string"}

# OpenAI-compatible base URL for Ollama or other endpoints
OLLAMA_URL = "https://ollama.com" #@param {type:"string"}

# Fallback URL and model used when the primary AI fails
OLLAMA_FALLBACK_URL = "https://ollama.com" #@param {type:"string"}
OLLAMA_FALLBACK_MODEL = "gemini-3-flash-preview:cloud" #@param {type:"string"}

# Gemini model used as fallback / when provider is gemini
GEMINI_MODEL = "gemini-3-flash-preview" #@param {type:"string"}


### ✂️ Clipping Settings

Control how many clips are generated and how they look.

- **Max Clips**: Total highlight clips to create (1–10).
- **Aspect Ratio**: Output shape. `9:16` is best for Shorts/Reels/TikTok.
- **Render Height**: Output resolution height. `1080` is standard.
- **Face Detector**: `mediapipe` (CPU friendly) or `yolo` (GPU).
- **Font Style**: Visual preset for subtitles.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — CLIPPING CONFIGURATION
# ═══════════════════════════════════════════════════════════════

# How many clips should the AI try to generate?
MAX_CLIPS = 5 #@param {type:"integer"}

# Output aspect ratio for the clips
ASPECT_RATIO = "9:16" #@param ["9:16", "16:9", "1:1", "3:4", "4:5"]

# Output height in pixels. Use 'source' to match the downloaded video height.
RENDER_HEIGHT = "1080" #@param ["source", "1080", "1440", "2160"]

# Font style preset for subtitles
FONT_STYLE = "HORMOZI" #@param ["DEFAULT", "HORMOZI", "STORYTELLER", "CINEMATIC"]

# Face tracking engine
FACE_DETECTOR = "mediapipe" #@param ["mediapipe", "yolo"]

# YOLO model size (only used if Face Detector is 'yolo')
YOLO_SIZE = "8m" #@param ["8n", "8s", "8m", "8n_v2", "9c"]


### 💬 Subtitle Settings

Configure on-screen text and transcription behavior.

- **Enable Subtitles**: Turn subtitle rendering on/off.
- **Use Karaoke Effect**: Highlight words as they are spoken.
- **Words Per Subtitle Group**: How many words appear in one subtitle chunk.
- **Whisper Model**: Speech-to-text model. `large-v3` is most accurate but slowest.
- **Transcript API**: Use YouTube's captions when available to skip Whisper entirely.
- **yt-dlp Subs**: Download YouTube auto-captions when possible.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — SUBTITLE & TRANSCRIPTION CONFIGURATION
# ═══════════════════════════════════════════════════════════════

# Turn subtitles on or off
ENABLE_SUBTITLES = True #@param {type:"boolean"}

# Highlight spoken words one-by-one (karaoke style)
USE_KARAOKE_EFFECT = True #@param {type:"boolean"}

# How many words appear in each subtitle group (1 = word-by-word)
WORDS_PER_SUBTITLE = 5 #@param {type:"integer"}

# Faster-Whisper model for transcription
WHISPER_MODEL = "large-v3" #@param {type:"string"}

# Whisper compute type
WHISPER_COMPUTE_TYPE = "float16" #@param ["float16", "float32", "int8"]

# Whisper device (auto will pick CUDA if available)
WHISPER_DEVICE = "auto" #@param ["auto", "cuda", "cpu"]

# Use YouTube Transcript API first to skip Whisper (YouTube URLs only)
USE_YT_TRANSCRIPT_API = False #@param {type:"boolean"}

# Download YouTube auto-captions with yt-dlp when available
USE_DLP_SUBS = True #@param {type:"boolean"}


### 📦 Output Settings

Choose where files are saved and how they are exported.

- **Output Folder**: Where final clips and metadata are written.
- **Video Preset**: Encoding speed/quality trade-off.
- **Scale Algorithm**: Resize quality for rendering.
- **CQ / CRF**: Quality values — lower means sharper but larger files.
- **Sharpen**: Apply a subtle sharpening filter.
- **Enable Cleanup**: Free GPU memory between pipeline stages (recommended on Colab).


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — OUTPUT CONFIGURATION
# ═══════════════════════════════════════════════════════════════

# Folder to save outputs. Leave empty to use the default 'outputs/' folder.
OUTPUT_FOLDER = "" #@param {type:"string"}

# Encoder preset: slower = smaller file, ultrafast = fastest
VIDEO_PRESET = "ultrafast" #@param ["ultrafast", "fast", "slow", "veryslow", "auto"]

# Scaling algorithm used during rendering
VIDEO_SCALE_ALGO = "bicubic" #@param ["lanczos", "bicubic", "bilinear", "area"]

# NVENC constant quality (lower = sharper, bigger file)
VIDEO_CQ = 23 #@param {type:"integer"}

# libx264 CRF (lower = sharper, bigger file)
VIDEO_CRF = 20 #@param {type:"integer"}

# Apply sharpening filter
VIDEO_SHARPEN = False #@param {type:"boolean"}

# Aggressively free GPU memory between stages (recommended on Colab)
ENABLE_COLAB_CLEANUP = True #@param {type:"boolean"}


### 🔧 Advanced Features (Optional)

Extra options for specific content styles. Most users can leave these off.

- **Hook V2**: Multi-hook intro with flash transitions.
- **Split Screen**: For podcasts with two speakers (requires HF_TOKEN).
- **Camera Switch**: Auto-switch between speakers (requires HF_TOKEN).
- **Disable B-roll / BGM / Hook**: Speed up and simplify output.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — ADVANCED FEATURES (OPTIONAL)
# ═══════════════════════════════════════════════════════════════

# Enable multi-hook intro (3-4 micro-hooks with transitions)
ENABLE_HOOK_V2 = False #@param {type:"boolean"}

# Hook teaser duration in seconds
HOOK_DURATION = 3 #@param {type:"integer"}

# Split-screen mode for 2-speaker podcasts (9:16 only, requires HF_TOKEN)
ENABLE_SPLIT_SCREEN = False #@param {type:"boolean"}

# What triggers split-screen layout changes
SPLIT_TRIGGER = "diarization" #@param ["diarization", "face"]

# Camera-switch mode for podcasts (requires HF_TOKEN)
ENABLE_CAMERA_SWITCH = False #@param {type:"boolean"}

# Disable specific features
DISABLE_BGM = True #@param {type:"boolean"}
DISABLE_BROLL = True #@param {type:"boolean"}
DISABLE_HOOK_GLITCH = False #@param {type:"boolean"}

# Segment trimming options
DISABLE_SEGMENT_TRIM = False #@param {type:"boolean"}
AGGRESSIVE_SILENCE_TRIM = False #@param {type:"boolean"}

# Hugging Face token (required for split-screen / camera-switch)
HF_TOKEN = "" #@param {type:"string"}


## ✅ Validate & Review

Run this cell after filling in the configuration above. It checks for required fields, API keys, valid ranges, and shows a summary table.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — VALIDATION & CONFIGURATION SUMMARY
# ═══════════════════════════════════════════════════════════════
from urllib.parse import urlparse
import re
import os

errors = []
warnings = []

# ── Input validation ──
video_url = VIDEO_URL.strip()
local_path = LOCAL_FILE_PATH.strip()
playlist_url = PLAYLIST_URL.strip()

if not video_url and not local_path and not playlist_url:
    errors.append("❌ Please provide at least one input: Video URL, Local File Path, or Playlist URL.")

if video_url and not (video_url.startswith("http://") or video_url.startswith("https://")):
    errors.append("❌ Video URL must start with http:// or https://")

if local_path and not os.path.exists(local_path):
    warnings.append(f"⚠️ Local file not found yet: {local_path}. It must exist before running the pipeline.")

# ── AI validation ──
if AI_PROVIDER == "ollama":
    if not OLLAMA_URL.strip():
        errors.append("❌ OLLAMA_URL is required when AI_PROVIDER is 'ollama'.")
    if not AI_MODEL.strip():
        errors.append("❌ AI_MODEL is required when AI_PROVIDER is 'ollama'.")
    if OLLAMA_URL.startswith("https://ollama.com") and not OLLAMA_API_KEY.strip():
        errors.append("❌ OLLAMA_API_KEY is required for Ollama Cloud (https://ollama.com). Get one at https://ollama.com/settings/keys")
elif AI_PROVIDER == "gemini":
    if not GOOGLE_API_KEY.strip():
        errors.append("❌ GOOGLE_API_KEY is required when AI_PROVIDER is 'gemini'.")
elif AI_PROVIDER == "nvidia":
    if not NVIDIA_API_KEY.strip():
        errors.append("❌ NVIDIA_API_KEY is required when AI_PROVIDER is 'nvidia'.")

# Always recommend a Gemini key for fallback
if AI_PROVIDER in ("ollama", "nvidia") and not GOOGLE_API_KEY.strip():
    warnings.append("⚠️ GOOGLE_API_KEY is empty. The pipeline will fail if the primary AI provider cannot fallback to Gemini.")

# ── Numeric range validation ──
if not (1 <= MAX_CLIPS <= 20):
    errors.append("❌ MAX_CLIPS must be between 1 and 20.")
if not (1 <= WORDS_PER_SUBTITLE <= 20):
    errors.append("❌ WORDS_PER_SUBTITLE must be between 1 and 20.")
if not (1 <= HOOK_DURATION <= 10):
    errors.append("❌ HOOK_DURATION must be between 1 and 10 seconds.")
if not (15 <= VIDEO_CQ <= 50):
    errors.append("❌ VIDEO_CQ must be between 15 and 50.")
if not (15 <= VIDEO_CRF <= 50):
    errors.append("❌ VIDEO_CRF must be between 15 and 50.")

# ── Feature compatibility ──
if ASPECT_RATIO != "9:16" and (ENABLE_SPLIT_SCREEN or ENABLE_CAMERA_SWITCH):
    warnings.append("⚠️ Split-screen and camera-switch only work reliably with 9:16 aspect ratio.")

if (ENABLE_SPLIT_SCREEN or ENABLE_CAMERA_SWITCH) and not HF_TOKEN.strip():
    errors.append("❌ HF_TOKEN is required for split-screen or camera-switch mode (Hugging Face pyannote model).")

# ── Summary table ──
summary = [
    ("Input", "Video URL", video_url or "—"),
    ("Input", "Local File", local_path or "—"),
    ("Input", "Playlist URL", playlist_url or "—"),
    ("Input", "Source Platform", SOURCE_PLATFORM),
    ("Input", "Source Height", SOURCE_HEIGHT),
    ("AI", "Provider", AI_PROVIDER),
    ("AI", "Model", AI_MODEL),
    ("AI", "Ollama URL", OLLAMA_URL),
    ("AI", "Fallback Model", OLLAMA_FALLBACK_MODEL),
    ("AI", "Google API Key", "✅ set" if GOOGLE_API_KEY.strip() else "—"),
    ("AI", "Ollama API Key", "✅ set" if OLLAMA_API_KEY.strip() else "—"),
    ("AI", "NVIDIA API Key", "✅ set" if NVIDIA_API_KEY.strip() else "—"),
    ("Clipping", "Max Clips", MAX_CLIPS),
    ("Clipping", "Aspect Ratio", ASPECT_RATIO),
    ("Clipping", "Render Height", RENDER_HEIGHT),
    ("Clipping", "Font Style", FONT_STYLE),
    ("Clipping", "Face Detector", FACE_DETECTOR),
    ("Clipping", "YOLO Size", YOLO_SIZE),
    ("Subtitles", "Enabled", ENABLE_SUBTITLES),
    ("Subtitles", "Karaoke Effect", USE_KARAOKE_EFFECT),
    ("Subtitles", "Words Per Group", WORDS_PER_SUBTITLE),
    ("Subtitles", "Whisper Model", WHISPER_MODEL),
    ("Subtitles", "Whisper Compute", WHISPER_COMPUTE_TYPE),
    ("Subtitles", "YouTube Transcript API", USE_YT_TRANSCRIPT_API),
    ("Subtitles", "yt-dlp Subs", USE_DLP_SUBS),
    ("Output", "Output Folder", OUTPUT_FOLDER or "outputs/"),
    ("Output", "Video Preset", VIDEO_PRESET),
    ("Output", "Scale Algorithm", VIDEO_SCALE_ALGO),
    ("Output", "Video CQ", VIDEO_CQ),
    ("Output", "Output CRF", VIDEO_CRF),
    ("Output", "Sharpen", VIDEO_SHARPEN),
    ("Output", "Colab Cleanup", ENABLE_COLAB_CLEANUP),
    ("Advanced", "Hook V2", ENABLE_HOOK_V2),
    ("Advanced", "Hook Duration", HOOK_DURATION),
    ("Advanced", "Split Screen", ENABLE_SPLIT_SCREEN),
    ("Advanced", "Split Trigger", SPLIT_TRIGGER),
    ("Advanced", "Camera Switch", ENABLE_CAMERA_SWITCH),
    ("Advanced", "Disable BGM", DISABLE_BGM),
    ("Advanced", "Disable B-roll", DISABLE_BROLL),
    ("Advanced", "Disable Hook Glitch", DISABLE_HOOK_GLITCH),
    ("Advanced", "Disable Segment Trim", DISABLE_SEGMENT_TRIM),
    ("Advanced", "Aggressive Silence Trim", AGGRESSIVE_SILENCE_TRIM),
    ("Advanced", "HF_TOKEN", "✅ set" if HF_TOKEN.strip() else "—"),
]

print("═" * 70)
print("📋 CONFIGURATION SUMMARY")
print("═" * 70)

current_section = None
for section, key, value in summary:
    if section != current_section:
        print(f"\n[{section}]")
        current_section = section
    print(f"  • {key:<28}: {value}")

if warnings:
    print("\n⚠️ WARNINGS")
    for w in warnings:
        print(f"  {w}")

if errors:
    print("\n❌ ERRORS — Please fix these before running the pipeline:")
    for e in errors:
        print(f"  {e}")
    raise SystemExit("Configuration validation failed.")

print("\n✅ Configuration looks good! Run the next cell to start the pipeline.")


## 🚀 Run Pipeline

Run this cell to start clipping. It builds the full command from the configuration above, executes the pipeline, and zips the outputs for download.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — RUN PIPELINE
# ═══════════════════════════════════════════════════════════════
import subprocess, sys, shutil, time, json
from pathlib import Path

# ── Build environment file ──
env_lines = []
if GOOGLE_API_KEY.strip(): env_lines.append(f"GOOGLE_API_KEY={GOOGLE_API_KEY.strip()}")
if OLLAMA_API_KEY.strip(): env_lines.append(f"OLLAMA_API_KEY={OLLAMA_API_KEY.strip()}")
if NVIDIA_API_KEY.strip(): env_lines.append(f"NVIDIA_API_KEY={NVIDIA_API_KEY.strip()}")
if HF_TOKEN.strip(): env_lines.append(f"HF_TOKEN={HF_TOKEN.strip()}")
if env_lines:
    Path(CLONE_DIR, ".env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")
    print("🔐 .env written.")

# ── Determine primary URL ──
url = VIDEO_URL.strip() or LOCAL_FILE_PATH.strip() or PLAYLIST_URL.strip()
if not url:
    raise ValueError("No input URL or file path provided. Please fill in the Input configuration.")

# ── Build CLI command ──
render_h = str(RENDER_HEIGHT)
source_h = str(SOURCE_HEIGHT)

cmd = [
    sys.executable, "main.py",
    "--url", url,
    "--source", SOURCE_PLATFORM,
    "--clips", str(MAX_CLIPS),
    "--ratio", ASPECT_RATIO,
    "--render-height", render_h,
    "--source-height", source_h,
    "--font-style", FONT_STYLE,
    "--face-detector", FACE_DETECTOR,
    "--yolo-size", YOLO_SIZE,
    "--ai-provider", AI_PROVIDER,
    "--gemini-model", GEMINI_MODEL,
    "--ollama-url", OLLAMA_URL,
    "--ollama-model", AI_MODEL,
    "--ollama-fallback-url", OLLAMA_FALLBACK_URL,
    "--ollama-fallback-model", OLLAMA_FALLBACK_MODEL,
    "--whisper-model", WHISPER_MODEL,
    "--whisper-device", WHISPER_DEVICE,
    "--whisper-compute-type", WHISPER_COMPUTE_TYPE,
    "--video-preset", VIDEO_PRESET,
    "--video-scale-algo", VIDEO_SCALE_ALGO,
    "--video-cq", str(VIDEO_CQ),
    "--video-crf", str(VIDEO_CRF),
    "--words-per-sub", str(WORDS_PER_SUBTITLE),
    "--hook-duration", str(HOOK_DURATION),
]

# Optional flags
def add_flag(flag, condition):
    if condition:
        cmd.append(flag)

add_flag("--colab-cleanup", ENABLE_COLAB_CLEANUP)
add_flag("--use-yt-transcript-api", USE_YT_TRANSCRIPT_API)
add_flag("--use-dlp-subs", USE_DLP_SUBS)
add_flag("--no-bgm", DISABLE_BGM)
add_flag("--no-broll", DISABLE_BROLL)
add_flag("--no-subs", not ENABLE_SUBTITLES)
add_flag("--no-karaoke", not USE_KARAOKE_EFFECT)
add_flag("--no-hook", DISABLE_HOOK_GLITCH)
add_flag("--camera-switch", ENABLE_CAMERA_SWITCH)
add_flag("--hook-v2", ENABLE_HOOK_V2)
add_flag("--no-segment-trim", DISABLE_SEGMENT_TRIM)
add_flag("--silence-trim", AGGRESSIVE_SILENCE_TRIM)
add_flag("--video-sharpen", VIDEO_SHARPEN)

if ENABLE_SPLIT_SCREEN:
    cmd.append("--split-screen")
    cmd += ["--split-trigger", SPLIT_TRIGGER]

# Note: the main pipeline currently writes to the fixed outputs/ folder under the repo.
# The OUTPUT_FOLDER setting is reserved for future support; for now, outputs go to outputs/.

print("🚀 Starting pipeline…")
print("Input:", url)
print("Command:", " ".join(cmd))
print("─" * 60)

start = time.time()
result = subprocess.run(cmd, cwd=CLONE_DIR)
if result.returncode != 0:
    print(f"❌ Pipeline exited with code {result.returncode}")
elapsed = time.time() - start

print("─" * 60)
print(f"⏱️ Total time: {elapsed/60:.1f} minutes")

# ── Collect outputs ──
out_dir = Path(CLONE_DIR) / "outputs"
if out_dir.exists():
    files = sorted(out_dir.glob("*_ready.mp4")) + sorted(out_dir.glob("*.jpg"))
    if files:
        print(f"\n📦 Output files ({len(files)}):")
        for f in files:
            print(f"   • {f.name} ({f.stat().st_size/1024/1024:.1f} MB)")
    if list(out_dir.iterdir()):
        zip_path = Path(CLONE_DIR) / "outputs.zip"
        shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(out_dir))
        print(f"\n🗜️ outputs.zip → {zip_path.stat().st_size/1024/1024:.1f} MB")
        try:
            from google.colab import files
            files.download(str(zip_path))
        except Exception:
            pass
else:
    print("⚠️ outputs/ directory not found.")
